# Entity and article entity representations

BT4222 Group 6 · representations 2/3. Run `3_01_news_embeddings.ipynb` first.

Validate the **training-release 100D entity coordinate system**, pool matched entity vectors into
article vectors, and distinguish missing annotations from missing lookup vectors. Relation embeddings
are excluded because the supplied news metadata does not include the corresponding graph triples.

Run top to bottom using the same Python environment as the other notebooks. All project code and settings are included below; no local Python module or settings file is imported. Upstream Parquet/NumPy artifacts remain the inputs shared between notebooks.

`REBUILD=False` reuses verified saved outputs; set it to `True` after changing this notebook's calculations/settings. Restarting VS Code clears variables, not saved datasets.

In [ ]:
# All project settings and implementation are in this notebook.
# Standard third-party packages come from requirements.txt.
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import google.colab
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass

# Edit DATA_ROOT to point to another project/data folder, if necessary.
DATA_ROOT = None
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/BT4222")]
if DATA_ROOT is not None:
    ROOT = Path(DATA_ROOT).expanduser().resolve()
elif os.environ.get("NEWSREC_ROOT"):
    ROOT = Path(os.environ["NEWSREC_ROOT"]).expanduser().resolve()
else:
    ROOT = next(
        (
            p
            for p in candidates
            if (p / "1_Data_Preparation/processed/news_catalogue.parquet").is_file()
        ),
        None,
    )
if ROOT is None:
    raise FileNotFoundError("Run data preparation first, or set DATA_ROOT above.")
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data and output root:", ROOT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import psutil
from dataclasses import dataclass

### Artifact checks and reporting

In [ ]:
NOTEBOOK_PATHS = (
    "3_Representations/3_01_news_embeddings.ipynb",
    "3_Representations/3_02_entity_embeddings.ipynb",
    "3_Representations/3_03_user_profiles.ipynb",
    "4_Feature_Engineering/4_01_build_ranking_features.ipynb",
)

In [ ]:
SPLITS = ("train", "validation", "test")

In [ ]:
VERSION = "newsrec-v1"

In [ ]:
@dataclass(frozen=True)
class Settings:
    seed: int = 4222
    text_dim: int = 64
    max_features: int = 50000
    min_df: int = 2
    title_weight: float = 0.7
    title_tokens: int = 30
    entity_dim: int = 100
    history_cap: int = 50
    snapshot_order_verified: bool = False
    recent_count: int = 10
    recent_hours: int = 24
    interest_count: int = 3
    ctr_strength: float = 20.0
    half_life_min_exposures: int = 100
    negative_count: int = 4
    update_mode: str = "prequential"
    context_features: bool = False
    parquet_batch_rows: int = 100000
    output_batch_rows: int = 100000

    def __post_init__(self):
        for key in (
            "text_dim",
            "max_features",
            "min_df",
            "title_tokens",
            "entity_dim",
            "history_cap",
            "recent_count",
            "recent_hours",
            "interest_count",
            "half_life_min_exposures",
            "negative_count",
            "parquet_batch_rows",
            "output_batch_rows",
        ):
            if getattr(self, key) <= 0:
                raise ValueError(f"{key} must be positive")
        if not 0 <= self.title_weight <= 1 or self.ctr_strength <= 0:
            raise ValueError("Invalid fusion/smoothing weights")
        if self.update_mode not in ("prequential", "frozen"):
            raise ValueError("update_mode must be prequential or frozen")

    def representation_fingerprint(self):
        fields = (
            "seed",
            "text_dim",
            "max_features",
            "min_df",
            "title_weight",
            "title_tokens",
            "entity_dim",
        )
        values = {name: getattr(self, name) for name in fields}
        return hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()

    def fingerprint(self):
        return hashlib.sha256(
            json.dumps(asdict(self), sort_keys=True).encode()
        ).hexdigest()

In [ ]:
DEFAULT_SETTINGS = Settings()

In [ ]:
# Edit settings here. Keep representation dimensions/vocabulary settings
# consistent across the notebooks that consume those artifacts.
SETTINGS = Settings(
    seed=4222,
    text_dim=64,
    max_features=50000,
    min_df=2,
    title_weight=0.7,
    title_tokens=30,
    entity_dim=100,
    history_cap=50,
    snapshot_order_verified=False,
    recent_count=10,
    recent_hours=24,
    interest_count=3,
    ctr_strength=20.0,
    half_life_min_exposures=100,
    negative_count=4,
    update_mode="prequential",
    context_features=False,
    parquet_batch_rows=100000,
    output_batch_rows=100000,
)
REBUILD = (
    False  # False: reuse verified outputs; True: regenerate this notebook's outputs
)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str) + "\n")

In [ ]:
def normalise(values):
    values = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(values, axis=-1, keepdims=True)
    return np.divide(values, norms, out=np.zeros_like(values), where=norms > 1e-12)

In [ ]:
def manifest(root, cfg, sources, outputs, **extra):
    root = Path(root)
    try:
        revision = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        revision = None
    return dict(
        version=VERSION,
        created_utc=pd.Timestamp.now(tz="UTC").isoformat(),
        config=asdict(cfg),
        config_hash=cfg.fingerprint(),
        representation_config_hash=cfg.representation_fingerprint(),
        git_revision=revision,
        python=platform.python_version(),
        libraries={
            n: importlib.metadata.version(n)
            for n in ("numpy", "pandas", "pyarrow", "polars", "scikit-learn")
        },
        implementation_hashes={
            relative: sha256(root / relative)
            for relative in NOTEBOOK_PATHS
            if (root / relative).is_file()
        },
        sources={str(Path(p).relative_to(root)): sha256(p) for p in sources},
        outputs={str(Path(p).relative_to(root)): sha256(p) for p in outputs},
        output_bytes=sum((Path(p).stat().st_size for p in outputs)),
        **extra,
    )

In [ ]:
@contextmanager
def measure(root, stage, **details):
    import threading

    process = psutil.Process()
    peak = [process.memory_info().rss]
    stop = threading.Event()

    def monitor():
        while not stop.wait(0.05):
            peak[0] = max(peak[0], process.memory_info().rss)

    worker = threading.Thread(target=monitor, daemon=True)
    worker.start()
    start = time.perf_counter()
    try:
        yield
    finally:
        peak[0] = max(peak[0], process.memory_info().rss)
        stop.set()
        worker.join()
        path = Path(root) / "results" / "representation_cost.csv"
        path.parent.mkdir(parents=True, exist_ok=True)
        row = dict(
            stage=stage,
            wall_seconds=time.perf_counter() - start,
            sampled_peak_process_rss_mb=peak[0] / 1000000.0,
            platform=platform.platform(),
            cpu=platform.processor() or platform.machine(),
            logical_cpus=psutil.cpu_count(),
            ram_gb=psutil.virtual_memory().total / 1000000000.0,
            **details,
        )
        pd.DataFrame([row]).to_csv(
            path, mode="a", header=not path.exists(), index=False
        )

In [ ]:
def upsert_report(root, filename, frame, stage):
    path = Path(root) / "results" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = frame.copy()
    frame["stage"] = stage
    if path.exists():
        old = pd.read_csv(path)
        frame = pd.concat([old[old.stage != stage], frame], ignore_index=True)
    frame.to_csv(path, index=False)

In [ ]:
def require_inputs(root):
    p = Path(root) / "1_Data_Preparation" / "processed"
    files = [p / "news_catalogue.parquet", p / "users.parquet"]
    files += [
        p / f"{kind}_{split}.parquet"
        for kind in ("impressions", "candidates")
        for split in SPLITS
    ]
    missing = [str(f) for f in files if not f.exists()]
    if missing:
        raise FileNotFoundError(
            "Run 1_Data_Preparation/clean_and_validate_data.ipynb first. Missing:\n"
            + "\n".join(missing)
        )
    return p

### Article representation and availability logic

In [ ]:
STATIC_COLUMNS = [
    "news_id",
    "news_code",
    "category",
    "subcategory",
    "title",
    "abstract",
    "title_entities",
    "abstract_entities",
]

In [ ]:
def artifact_dir(root):
    path = Path(root) / "3_Representations" / "artifacts"
    path.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
def read_catalogue(root):
    path = Path(root) / "1_Data_Preparation" / "processed" / "news_catalogue.parquet"
    df = pd.read_parquet(path)
    required = STATIC_COLUMNS + ["in_news_train"]
    if not set(required) <= set(df):
        raise ValueError(f"Catalogue lacks {set(required) - set(df)}")
    df = df.sort_values("news_code").reset_index(drop=True)
    if df.news_code.duplicated().any() or df.news_id.duplicated().any():
        raise ValueError("Article IDs must be unique")
    return df

In [ ]:
def read_entity_lookup(path, dim):
    ids, vectors = ([], [])
    seen = set()
    with Path(path).open(encoding="utf-8-sig") as fh:
        for line_no, line in enumerate(fh, 1):
            parts = line.split()
            if not parts:
                continue
            if len(parts) != dim + 1:
                raise ValueError(f"{path}:{line_no}: expected ID + {dim} coordinates")
            if parts[0] in seen:
                raise ValueError(f"Duplicate entity {parts[0]}")
            vector = np.asarray(parts[1:], np.float32)
            if not np.isfinite(vector).all():
                raise ValueError(f"Nonfinite entity {parts[0]}")
            ids.append(parts[0])
            vectors.append(vector)
            seen.add(parts[0])
    if not ids:
        raise ValueError("Entity file is empty")
    return (ids, np.asarray(vectors, np.float32))

In [ ]:
def parse_entities(value):
    if (
        value is None
        or (isinstance(value, float) and np.isnan(value))
        or (not str(value).strip())
    ):
        return ([], "missing")
    try:
        data = json.loads(value)
    except (ValueError, TypeError):
        return ([], "invalid")
    if not isinstance(data, list) or any((not isinstance(e, dict) for e in data)):
        return ([], "invalid")
    ids = sorted({str(e["WikidataId"]) for e in data if e.get("WikidataId")})
    return (ids, "present" if data else "empty")

In [ ]:
def build_entities(root, cfg=DEFAULT_SETTINGS):
    require_inputs(root)
    out = artifact_dir(root)
    news_manifest = json.loads((out / "news_manifest.json").read_text())
    if news_manifest["representation_config_hash"] != cfg.representation_fingerprint():
        raise ValueError("Rebuild news representations with this configuration first")
    with measure(root, "entity"):
        index = pd.read_parquet(out / "article_index.parquet")
        if sha256(out / "article_index.parquet") != news_manifest["article_index_hash"]:
            raise ValueError("Article index changed; rebuild text representations")
        source = Path(root) / "train" / "entity_embedding.vec"
        ids, raw = read_entity_lookup(source, cfg.entity_dim)
        lookup = {key: i + 1 for i, key in enumerate(ids)}
        vectors = np.vstack([np.zeros((1, cfg.entity_dim), np.float32), normalise(raw)])
        np.save(out / "entity_vectors.npy", vectors)
        pd.DataFrame(
            {"entity_id": ids, "entity_row": np.arange(1, len(ids) + 1)}
        ).to_parquet(out / "entity_index.parquet", index=False)
        article = np.zeros((len(index) + 1, cfg.entity_dim), np.float32)
        links, quality, article_sets = ([], [], [])
        for row in index.itertuples(index=False):
            title_ids, ts = parse_entities(row.title_entities)
            abstract_ids, aps = parse_entities(row.abstract_entities)
            all_ids = sorted(set(title_ids) | set(abstract_ids))
            article_sets.append(set(all_ids))
            matched = [
                lookup[x]
                for x in all_ids
                if x in lookup and np.linalg.norm(vectors[lookup[x]]) > 0
            ]
            if matched:
                article[row.article_row] = normalise(vectors[matched].mean(axis=0))
            quality.append(
                {
                    "news_code": row.news_code,
                    "title_entity_status": ts,
                    "abstract_entity_status": aps,
                    "n_title_entities": len(title_ids),
                    "n_abstract_entities": len(abstract_ids),
                    "n_annotated_entities": len(all_ids),
                    "n_valid_entities": len(matched),
                    "entity_coverage": len(matched) / max(1, len(all_ids)),
                    "has_entity_vector": bool(
                        np.linalg.norm(article[row.article_row]) > 0
                    ),
                }
            )
            for key in all_ids:
                links.append(
                    {
                        "news_code": row.news_code,
                        "entity_id": key,
                        "entity_row": lookup.get(key, 0),
                        "in_title": key in title_ids,
                        "in_abstract": key in abstract_ids,
                    }
                )
        np.save(out / "article_entity100.npy", article)
        pd.DataFrame(quality).to_parquet(out / "entity_quality.parquet", index=False)
        pd.DataFrame(
            links,
            columns=["news_code", "entity_id", "entity_row", "in_title", "in_abstract"],
        ).to_parquet(out / "article_entity_links.parquet", index=False)
        catalogue = read_catalogue(root)
        coverage = []
        for split in SPLITS:
            mask = catalogue[f"in_news_{split}"].to_numpy(bool)
            observed = set().union(*(article_sets[i] for i in np.flatnonzero(mask)))
            valid_ids = {
                x
                for x in observed
                if x in lookup and np.linalg.norm(vectors[lookup[x]]) > 0
            }
            for rep, n, k in (
                ("entity_ids", len(observed), len(valid_ids)),
                (
                    "article_entity",
                    int(mask.sum()),
                    int((np.linalg.norm(article[1:][mask], axis=1) > 0).sum()),
                ),
            ):
                coverage.append(
                    {
                        "split": split,
                        "representation": rep,
                        "cohort": "split_catalogue",
                        "denominator": n,
                        "valid": k,
                        "fallback": n - k,
                        "valid_fraction": k / max(1, n),
                    }
                )
        upsert_report(
            root, "representation_coverage.csv", pd.DataFrame(coverage), "entity"
        )
        compatibility = {
            "status": "dev lookup absent; canonical train space used throughout"
        }
        dev = Path(root) / "test" / "entity_embedding.vec"
        sources = [source, out / "article_index.parquet"]
        if dev.exists():
            dev_ids, dev_raw = read_entity_lookup(dev, cfg.entity_dim)
            dev_map = {k: i for i, k in enumerate(dev_ids)}
            shared = sorted(set(ids) & set(dev_ids))
            max_diff = max(
                (
                    float(np.max(np.abs(raw[lookup[k] - 1] - dev_raw[dev_map[k]])))
                    for k in shared
                ),
                default=None,
            )
            compatibility = {
                "shared_ids": len(shared),
                "max_coordinate_difference": max_diff,
                "dev_only_ids": len(set(dev_ids) - set(ids)),
                "policy": "audit only; no dev-only vectors merged",
            }
            sources.append(dev)
        outputs = [
            out / name
            for name in (
                "entity_vectors.npy",
                "entity_index.parquet",
                "article_entity100.npy",
                "entity_quality.parquet",
                "article_entity_links.parquet",
            )
        ]
        write_json(
            out / "entity_manifest.json",
            manifest(
                root,
                cfg,
                sources,
                outputs,
                article_index_hash=sha256(out / "article_index.parquet"),
                canonical_space="train",
                zero_lookup_vectors=int((np.linalg.norm(raw, axis=1) == 0).sum()),
                dev_compatibility=compatibility,
            ),
        )
    return pd.DataFrame(coverage)

In [ ]:
def saved_stage_ready(root, settings, stage):
    """Reuse disk outputs only when settings, inputs and output hashes agree."""
    path = Path(root) / "3_Representations/artifacts" / f"{stage}_manifest.json"
    if not path.exists():
        return False
    meta = json.loads(path.read_text())
    key = "config_hash" if stage == "profiles" else "representation_config_hash"
    expected = (
        settings.fingerprint()
        if stage == "profiles"
        else settings.representation_fingerprint()
    )
    if meta[key] != expected:
        raise ValueError(
            "Saved settings differ. Set REBUILD=True and rerun dependent notebooks."
        )
    for kind in ("sources", "outputs"):
        for relative, expected_hash in meta[kind].items():
            file = Path(root) / relative
            if not file.exists() or sha256(file) != expected_hash:
                raise ValueError(
                    f"Saved {stage} artifacts are stale or incomplete: {relative}. Set REBUILD=True."
                )
    return True

## Build or load saved outputs

## 1. Parse and pool

Parse JSON; deduplicate Wikidata IDs across title and abstract; normalise each matched entity vector,
mean-pool, then normalise the article vector. Empty/unmatched articles use zero plus a validity mask.
The dev lookup is audited for coordinate differences, never merged automatically into the train space.
Zero vectors, malformed rows, nonfinite coordinates and duplicate entity IDs are checked explicitly.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "entity"):
    coverage = build_entities(ROOT, SETTINGS)
else:
    print("Loaded saved entity representations.")
    report = pd.read_csv(ROOT / "results/representation_coverage.csv")
    coverage = report[report.stage.eq("entity")]
display(coverage)

## 2. Audit coverage at the right denominator

Entity-ID lookup coverage, article-vector coverage and user-profile coverage are different quantities.
The preparation report's test lookup coverage uses the dev file; this notebook reports the canonical
train lookup for every split. It separately records whether the dev coordinates match.

In [ ]:
quality = pd.read_parquet(ARTIFACTS / "entity_quality.parquet")
display(quality.head())
display(
    quality.groupby(["title_entity_status", "abstract_entity_status"])
    .size()
    .rename("articles")
    .reset_index()
)
meta = json.loads((ARTIFACTS / "entity_manifest.json").read_text())
display(meta["dev_compatibility"])
vectors = np.load(ARTIFACTS / "article_entity100.npy", mmap_mode="r")
assert vectors.shape == (len(quality) + 1, SETTINGS.entity_dim)
assert np.isfinite(vectors).all() and np.all(vectors[0] == 0)
assert np.array_equal(
    np.linalg.norm(vectors[1:], axis=1) > 0, quality.has_entity_vector.to_numpy()
)
print("Article entity matrix MB:", vectors.nbytes / 1e6)

## Handoff

`article_entity100.npy` shares the article-row map with the text vectors. Entity and text coordinates
remain separate. `article_entity_links.parquet` supplies annotation links (including unmatched IDs)
for later graph construction; these annotation links are not evidence that an embedding exists.
The following notebook produces supplied-only entity user profiles for the clustering owner.